# Build Your Own RAG System with Ollama and Python

**What you will build:** a small question-answering system that reads from documents *you* choose and answers questions using a language model running on your own computer.

**The key idea (and why it matters here):** everything in this notebook runs locally through Ollama. Once the models are downloaded, you do not need an internet connection, and the text you feed the system never leaves your laptop. That makes this a working example of keeping data under your own control.

**Who this is for:** you do not need to be a programmer. Read the short explanations, then run each grey code cell from top to bottom (press Shift+Enter, or use the Run button).

**Time:** about 45 to 60 minutes.

## Step 1. Set up Ollama and the models

Do these two things once before running the code below.

**1. Install Ollama.** Download it from https://ollama.com/download and install it like any other app. Make sure the Ollama app is running (you should see it in your menu bar or system tray).

**2. Download the two models we will use.** Open a terminal (Terminal on Mac, Command Prompt or PowerShell on Windows) and run:

```
ollama pull llama3.2
ollama pull nomic-embed-text
```

- `llama3.2` is the language model that writes answers. If your laptop has limited memory, use `llama3.2:1b` instead (it is smaller and faster, but less capable).
- `nomic-embed-text` is a small model that turns text into numbers so we can compare meanings.

The convenience cell below can also download them for you. It only needs to run once.

In [ ]:
# Optional: download the models from inside the notebook (only needed once).
# If you already ran the `ollama pull` commands in a terminal, you can skip this.
!ollama pull llama3.2
!ollama pull nomic-embed-text

Now install the Python packages we need:
- `ollama` — talk to Ollama from Python
- `numpy` — fast array math for similarity scores
- `h5py` — save and load embeddings to disk in the HDF5 format (so we never have to re-compute them)
- `faiss-cpu` — Facebook AI Similarity Search, a library that finds nearest neighbours in large embedding collections very quickly

In [2]:
# Install the Python packages into this notebook's environment.
%pip install ollama numpy h5py faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.6 kB/s  0:00:42 eta 0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 72.3 kB/s  0:01:07m0:00:0100:05m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ollama]2m1/3 [faiss-cpu]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2. Talk to the model from Python

Let's make sure everything is connected, then send the model its first message.

In [15]:
import ollama
import numpy as np

# The two models we will use. Both run locally through Ollama.
CHAT_MODEL  = "gemma4:latest"           # try "llama3.2:1b" if your laptop has limited memory
EMBED_MODEL = "nomic-embed-text"   # turns text into numbers (an "embedding")

# Quick check that Ollama is running and reachable from Python.
try:
    print("Available models in Ollama:")
    for model in ollama.list().models:
        print(model.model)
except Exception as e:
    print("Could not reach Ollama.")
    print("Make sure the Ollama application is running, or run 'ollama serve' in a terminal.")
    print("Details:", e)

Available models in Ollama:
nomic-embed-text:latest
gemma4:latest
glm-ocr:latest
minicpm-v4.6:latest


In [16]:
# Send the model a question and print its answer.
print("Loading model...")
response = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is a large language model?"}
    ],
)

print(response["message"]["content"])

Loading model...
A large language model is a sophisticated AI program trained on enormous amounts of text data that learns patterns in human language, enabling it to predict, understand, and generate remarkably coherent and contextually relevant text responses.


You can also stream the answer so it appears word by word, the way a chat app does.

In [17]:
print("Loading model...")
stream = ollama.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Explain what an embedding is, in two sentences."}],
    stream=True,
)

print("Responding...")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print()

Loading model...
Responding...
An embedding is a mathematical technique that maps complex or discrete data points (such as words, images, or users) into a dense, continuous vector space. In this space, the geometric proximity between two vectors reflects the underlying similarity or relationship between the original data items they represent—for example, similar words will have close vector coordinates.


## Step 3. Embeddings: turning meaning into numbers

A RAG system needs a way to find which document is relevant to a question. It does this with **embeddings**.

An embedding is just a list of numbers that represents the *meaning* of a piece of text. Texts that mean similar things get embeddings that point in similar directions, even if they use completely different words.

Let's turn one sentence into an embedding and look at it.

In [18]:
def embed(text):
    # Returns a list of numbers (a vector) representing the meaning of the text.
    return ollama.embeddings(model=EMBED_MODEL, prompt=text)["embedding"]

vector = embed("Communities have the right to control data collected about them.")
print("This embedding has", len(vector), "numbers.")
print("First 8 numbers:", [round(x, 3) for x in vector[:8]])

This embedding has 768 numbers.
First 8 numbers: [0.809, 0.24, -3.801, -0.561, 1.472, 0.029, 0.516, 0.508]


To find relevant documents, we measure how *close* two embeddings are using **cosine similarity**. A score near 1 means very similar in meaning; a score near 0 means unrelated.

Notice below that two sentences about data sovereignty score high even though they share almost no words, while the sentence about model training scores low.

In [19]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

a = embed("Communities have the right to control data collected about them.")
b = embed("Indigenous peoples should govern how their information is stored and used.")
c = embed("The neural network was trained on a large corpus of public text.")

print("Similar sentences (both about data sovereignty): ", round(cosine_similarity(a, b), 3))
print("Unrelated sentence (about model training):       ", round(cosine_similarity(a, c), 3))

Similar sentences (both about data sovereignty):  0.678
Unrelated sentence (about model training):        0.453


## Step 4. Chunking: splitting documents into searchable pieces

Before we build a knowledge base, we need to think about *how* to feed documents into it.

**The problem with long documents.** An embedding is a single fixed-size vector, no matter how long the text. If you embed a 10-page report, that single vector has to represent everything in it — and when a question only relates to one paragraph, the match will be blurry. Retrieval becomes vague.

**The solution: chunking.** Split long documents into shorter passages (chunks) before embedding. Each chunk gets its own vector, so retrieval can pinpoint the exact passage that matches the question.

There are several ways to chunk. We will look at three:

| Strategy | How it works | Good for |
|---|---|---|
| Fixed size | Split every N words | Simple; baseline approach |
| Fixed size with overlap | Same, but each chunk shares some words with the next | Avoids cutting a thought in half |
| Sentence-based | Split on `.` / `?` / `!` then group sentences | Keeps ideas intact |

There is no single best answer. Overlap is almost always worth adding because it prevents a key sentence from being split across two chunks where it ends up weak in both.

In [20]:
import re

def chunk_fixed(text, max_words=60):
    """Split text into non-overlapping word windows."""
    words = text.split()
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

def chunk_overlap(text, max_words=60, overlap=15):
    """Split text into overlapping word windows."""
    words  = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + max_words]))
        i += max_words - overlap
    return chunks

def chunk_sentences(text, sentences_per_chunk=3):
    """Group text into chunks of N sentences."""
    sentences = [s.strip() for s in re.split(r'(?<=[.?!])\s+', text) if s.strip()]
    return [
        " ".join(sentences[i:i + sentences_per_chunk])
        for i in range(0, len(sentences), sentences_per_chunk)
    ]

# --- Demo on a short paragraph ---
sample = (
    "Indigenous data sovereignty is the right of Indigenous peoples to govern the "
    "collection, ownership, and application of data about their nations, peoples, "
    "lands, and resources. It is grounded in Indigenous peoples' right to "
    "self-determination. Around the world, Indigenous communities are building "
    "their own data infrastructure so that research about them serves their own "
    "priorities rather than outside interests. This includes deciding what is "
    "collected, who can see it, how long it is kept, and under what conditions "
    "it may be shared."
)

print("=== Fixed (60 words, no overlap) ===")
for i, c in enumerate(chunk_fixed(sample, max_words=60)):
    print(f"  Chunk {i+1}: {c[:80]}...")

print()
print("=== Overlap (60 words, 15 overlap) ===")
for i, c in enumerate(chunk_overlap(sample, max_words=60, overlap=15)):
    print(f"  Chunk {i+1}: {c[:80]}...")

print()
print("=== Sentence-based (3 sentences per chunk) ===")
for i, c in enumerate(chunk_sentences(sample, sentences_per_chunk=3)):
    print(f"  Chunk {i+1}: {c[:80]}...")

=== Fixed (60 words, no overlap) ===
  Chunk 1: Indigenous data sovereignty is the right of Indigenous peoples to govern the col...
  Chunk 2: deciding what is collected, who can see it, how long it is kept, and under what ...

=== Overlap (60 words, 15 overlap) ===
  Chunk 1: Indigenous data sovereignty is the right of Indigenous peoples to govern the col...
  Chunk 2: so that research about them serves their own priorities rather than outside inte...

=== Sentence-based (3 sentences per chunk) ===
  Chunk 1: Indigenous data sovereignty is the right of Indigenous peoples to govern the col...
  Chunk 2: This includes deciding what is collected, who can see it, how long it is kept, a...


**Why does the chunk size matter in practice?**

The cell below embeds a longer passage two ways — as one big block and as sentence-level chunks — and checks which approach retrieves the most relevant piece for a specific question. You will often see that the chunked version scores higher because the query only needs to match a small section of the text.

In [23]:
long_passage = (
    "AI systems are trained on massive datasets assembled from the internet and other sources. "
    "Much of this data was collected without the knowledge or consent of the people it describes. "
    "Indigenous oral histories, cultural knowledge, and community records have often been digitized and published online without permission from the communities they belong to. "
    "The OCAP principles — Ownership, Control, Access, and Possession — were developed by the First Nations Information Governance Centre and assert that First Nations communities have collective ownership of their cultural knowledge and data. "
    "Some nations are establishing community data trusts, legal structures that hold data on behalf of the community and require collective consent before any sharing or use. "
    "Existing privacy laws focus on individual rights and have been slow to recognize the collective dimension of Indigenous data sovereignty."
)

query = "What are the OCAP principles?"
query_vec = embed(query)

# As one block
whole_score = cosine_similarity(query_vec, embed(long_passage))
print(f"Whole passage score: {whole_score:.3f}")

# As sentence chunks
chunks = chunk_sentences(long_passage, sentences_per_chunk=1)
best_score, best_chunk = 0, ""
for chunk in chunks:
    score = cosine_similarity(query_vec, embed(chunk))
    if score > best_score:
        best_score, best_chunk = score, chunk

print(f"Best chunk score:   {best_score:.3f}")
print(f"Best chunk text:    {best_chunk}")

Whole passage score: 0.580
Best chunk score:   0.781
Best chunk text:    The OCAP principles — Ownership, Control, Access, and Possession — were developed by the First Nations Information Governance Centre and assert that First Nations communities have collective ownership of their cultural knowledge and data.


## Step 5. Build a knowledge base

This is the collection of documents the system is allowed to read. For now we use a few short passages about the tools in this workshop. Later you will replace these with your own.

We embed every document once, then **save the embeddings to disk in HDF5 format** using the `h5py` library. HDF5 is a binary file format designed for large numerical arrays. The advantage: if you restart this notebook tomorrow, you can load the saved embeddings in milliseconds instead of waiting for the model to recompute them.

The pattern is:
1. Check whether a cache file already exists.
2. If yes, load embeddings from it.
3. If no, embed every document, then save to the cache.

In [24]:
import h5py
import json
import os

documents = [
    "Retrieval-Augmented Generation (RAG) combines a language model with a search step. Instead of relying only on what the model memorized during training, the system first retrieves relevant documents from a collection you control, then asks the model to answer using those documents. This grounds answers in your own sources and reduces made-up information.",
    "An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar meanings have embeddings that point in similar directions, so a computer can find which stored document is most relevant to a question even when they share no exact words.",
    "Ollama is a tool that runs large language models directly on your own computer. Because the model runs locally, the text you send it never leaves your machine, and you do not need an internet connection once the model is downloaded.",
    "Cosine similarity measures how close two embeddings are by looking at the angle between them. A value near 1 means the two texts are very similar in meaning, and a value near 0 means they are unrelated. RAG uses it to rank stored documents against a question.",
    "Running AI models locally matters for data sovereignty. When a community runs its own models on its own hardware, it keeps control over its data and decides who can access it, rather than sending sensitive material to servers owned by outside companies.",
    "Te Hiku Media, a Maori organization in New Zealand, built a speech recognition system for the Maori language. They release their data under a Kaitiakitanga License, which treats data as cared for under guardianship rather than owned, and ensures any benefit derived from the data flows back to the people it came from.",
    "A vector store keeps the embeddings of all your documents so they can be searched quickly. In a small project, a simple list in memory is enough. For larger collections, tools such as FAISS store and search many embeddings efficiently without scanning every vector one by one.",
    "Chunking means splitting long documents into smaller passages before embedding them. Smaller chunks make retrieval more precise, because the system can return just the relevant passage instead of an entire document. Overlapping chunks prevent key sentences from being lost at boundaries.",
]

CACHE_FILE = "embeddings_cache.h5"

def save_embeddings_hdf5(path, docs, embeddings):
    with h5py.File(path, "w") as f:
        f.create_dataset("embeddings", data=np.array(embeddings, dtype=np.float32))
        # Store document texts as a JSON string (HDF5 handles text less naturally than numpy arrays)
        f.attrs["documents"] = json.dumps(docs)
    print(f"Saved {len(docs)} embeddings to {path}")

def load_embeddings_hdf5(path):
    with h5py.File(path, "r") as f:
        embeddings = f["embeddings"][:].tolist()   # numpy → Python list
        docs = json.loads(f.attrs["documents"])
    print(f"Loaded {len(docs)} embeddings from {path}")
    return docs, embeddings

if os.path.exists(CACHE_FILE):
    documents, doc_embeddings = load_embeddings_hdf5(CACHE_FILE)
else:
    print(f"Embedding {len(documents)} documents. This runs locally and may take a few seconds...")
    doc_embeddings = [embed(doc) for doc in documents]
    save_embeddings_hdf5(CACHE_FILE, documents, doc_embeddings)

print("Knowledge base is ready:", len(documents), "documents,", len(doc_embeddings[0]), "-dimensional embeddings.")

Embedding 8 documents. This runs locally and may take a few seconds...
Saved 8 embeddings to embeddings_cache.h5
Knowledge base is ready: 8 documents, 768 -dimensional embeddings.


You can inspect the HDF5 file at any time to see exactly what is stored in it.

In [25]:
with h5py.File(CACHE_FILE, "r") as f:
    emb = f["embeddings"]
    print("Dataset shape:", emb.shape)          # (n_docs, embedding_dim)
    print("Data type:    ", emb.dtype)
    print("File size (bytes):", os.path.getsize(CACHE_FILE))
    docs_stored = json.loads(f.attrs["documents"])
    print("\nFirst stored document (first 80 chars):")
    print(" ", docs_stored[0][:80], "...")

Dataset shape: (8, 768)
Data type:     float32
File size (bytes): 30720

First stored document (first 80 chars):
  Retrieval-Augmented Generation (RAG) combines a language model with a search ste ...


## Step 6. Retrieval with FAISS

With a handful of documents, comparing the query to every embedding in a loop is instant. But if you have ten thousand documents, that sequential scan starts to slow down.

**FAISS** (Facebook AI Similarity Search) builds an index that finds the nearest embeddings to a query in sub-linear time. For small collections the speed difference is invisible; for large ones it can be the difference between a response in milliseconds and one that takes seconds.

We will use `IndexFlatIP`, which computes **inner product** (dot product). When all vectors are normalised to unit length first, inner product equals cosine similarity — so the ranking is identical to what we computed by hand in Step 3, just faster.

**Building the index is a one-time step.** After that, every search is a single call.

In [26]:
import faiss

def build_faiss_index(embeddings):
    """Build a FAISS flat inner-product index from a list of embedding vectors."""
    matrix = np.array(embeddings, dtype=np.float32)
    # L2-normalise so that inner product == cosine similarity
    faiss.normalize_L2(matrix)
    dim   = matrix.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(matrix)
    return index, matrix

faiss_index, normalized_matrix = build_faiss_index(doc_embeddings)
print(f"FAISS index built. Vectors stored: {faiss_index.ntotal}, dimension: {normalized_matrix.shape[1]}")

FAISS index built. Vectors stored: 8, dimension: 768


Given a question, we embed it, normalise it the same way, and ask FAISS for the top-k matches. FAISS returns the indices and scores directly — no manual loop required.

In [27]:
def retrieve(query, k=3):
    """Return the k most relevant (document, score) pairs for a query using FAISS."""
    query_vec = np.array([embed(query)], dtype=np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = faiss_index.search(query_vec, k)
    return [(documents[i], float(scores[0][rank])) for rank, i in enumerate(indices[0])]

# Try it: which documents are most relevant to this question?
for doc, score in retrieve("Why does running models on my own computer protect my data?"):
    print(round(score, 3), "->", doc[:80], "...")

0.708 -> Running AI models locally matters for data sovereignty. When a community runs it ...
0.61 -> Ollama is a tool that runs large language models directly on your own computer.  ...
0.508 -> A vector store keeps the embeddings of all your documents so they can be searche ...


**Sanity check.** Let's confirm the FAISS results match what the manual cosine loop would return.

In [28]:
question = "What is data sovereignty?"

# FAISS retrieval
print("FAISS results:")
for doc, score in retrieve(question, k=2):
    print(f"  {score:.3f}  {doc[:80]}...")

# Manual cosine loop (reference)
query_vec = embed(question)
scores = [(cosine_similarity(query_vec, d), documents[i]) for i, d in enumerate(doc_embeddings)]
scores.sort(reverse=True)
print("\nManual cosine loop results:")
for score, doc in scores[:2]:
    print(f"  {score:.3f}  {doc[:80]}...")

FAISS results:
  0.719  Running AI models locally matters for data sovereignty. When a community runs it...
  0.572  Te Hiku Media, a Maori organization in New Zealand, built a speech recognition s...

Manual cosine loop results:
  0.719  Running AI models locally matters for data sovereignty. When a community runs it...
  0.572  Te Hiku Media, a Maori organization in New Zealand, built a speech recognition s...


## Step 7. Query reformulation: helping the retriever understand the question

There is a subtle mismatch baked into most RAG systems. Your knowledge base is full of **declarative sentences** — passages that state facts. But users ask **questions**. These live in different parts of the embedding space, even when they are about exactly the same topic.

**Example.** The question *"Why does temperature affect salmon eggs?"* and the answer *"Warmer river temperatures reduce the survival rate of salmon eggs"* mean the same thing, but their embeddings point in slightly different directions. The stored passage is a statement; the query is a question. Retrieval works better when they match in style.

**Two complementary techniques address this:**

| Technique | How it works | When to use it |
|---|---|---|
| **Query rewriting** | Ask the LLM to rephrase the question as a declarative statement | General-purpose; low cost |
| **HyDE** (Hypothetical Document Embeddings) | Ask the LLM to write a *short hypothetical answer*, then embed that | Best when questions are very different in style from stored documents |

Both approaches embed the *output of the LLM* as the query vector instead of the raw user question. The LLM output sounds more like a document, so FAISS finds a closer match.

The tradeoff: you pay one extra LLM call per query. For small local models like `llama3.2` this is fast, and the retrieval improvement is often worth it.

In [29]:
def rewrite_query(question):
    """Ask the LLM to rephrase a question as a declarative statement."""
    prompt = (
        "Rewrite the following question as a short declarative statement (one or two sentences). "
        "Do not answer the question — just rephrase it so it reads like a fact or explanation rather than a question. "
        "Output only the rewritten statement, nothing else.\n\n"
        "Question: " + question
    )
    response = ollama.chat(model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"].strip()

def hyde_query(question):
    """Ask the LLM to write a hypothetical answer passage, then use that as the query vector."""
    prompt = (
        "Write a short passage (two or three sentences) that would directly answer the question below. "
        "You are not expected to know the real answer — write a plausible, document-like passage. "
        "Output only the passage, nothing else.\n\n"
        "Question: " + question
    )
    response = ollama.chat(model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"].strip()

def retrieve_rewritten(question, k=3):
    rewritten = rewrite_query(question)
    print("Rewritten query:", rewritten)
    return retrieve(rewritten, k)

def retrieve_hyde(question, k=3):
    hypothesis = hyde_query(question)
    print("Hypothetical passage:", hypothesis)
    return retrieve(hypothesis, k)

Let's compare all three retrieval strategies side by side on a question phrased very differently from how its answer would be written in a document.

In [30]:
question = "What happens to my text when I use an online AI?"

print("=" * 60)
print("RAW QUERY (no reformulation)")
print("=" * 60)
for doc, score in retrieve(question, k=2):
    print(f"  {score:.3f}  {doc[:90]}...")

print()
print("=" * 60)
print("REWRITTEN QUERY (question → declarative statement)")
print("=" * 60)
for doc, score in retrieve_rewritten(question, k=2):
    print(f"  {score:.3f}  {doc[:90]}...")

print()
print("=" * 60)
print("HyDE (hypothetical answer passage used as query)")
print("=" * 60)
for doc, score in retrieve_hyde(question, k=2):
    print(f"  {score:.3f}  {doc[:90]}...")

RAW QUERY (no reformulation)
  0.625  Running AI models locally matters for data sovereignty. When a community runs its own mode...
  0.551  Retrieval-Augmented Generation (RAG) combines a language model with a search step. Instead...

REWRITTEN QUERY (question → declarative statement)
Rewritten query: Online AIs process your submitted text data.
  0.648  Running AI models locally matters for data sovereignty. When a community runs its own mode...
  0.569  Retrieval-Augmented Generation (RAG) combines a language model with a search step. Instead...

HyDE (hypothetical answer passage used as query)
Hypothetical passage: Your text input is temporarily processed on secure servers for real-time analysis and output generation. Depending on your account settings, this data may be stored and reviewed by our systems to improve model performance and ensure compliance with usage policies. We utilize aggregated and often anonymized datasets derived from interactions solely for refining the AI's u

We can wire reformulation directly into `ask` so the full RAG loop benefits automatically. The `strategy` parameter lets you switch between the three modes without touching anything else.

In [31]:
def ask(query, k=3, strategy="raw"):
    """
    Full RAG loop with optional query reformulation.

    strategy options:
      'raw'     — embed the question directly (baseline)
      'rewrite' — rewrite question as a declarative statement first
      'hyde'    — generate a hypothetical answer passage first (HyDE)
    """
    if strategy == "rewrite":
        retrieved = retrieve_rewritten(query, k)
    elif strategy == "hyde":
        retrieved = retrieve_hyde(query, k)
    else:
        retrieved = retrieve(query, k)

    context  = "\n".join("- " + doc for doc, score in retrieved)
    prompt   = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say you do not know.\n\n"
        "Context:\n" + context + "\n\n"
        "Question: " + query + "\n"
        "Answer:"
    )
    response = ollama.chat(model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"]

# Default: no reformulation (matches the original behaviour)
print(ask("What is the Kaitiakitanga License and what does it ensure?"))

The Kaitiakitanga License treats data as cared for under guardianship rather than owned, and it ensures any benefit derived from the data flows back to the people it came from.


## Step 8. Generation: the full RAG loop

Now we put it together. The `ask` function defined in the previous step:

1. optionally reformulates the query (Step 7),
2. retrieves the most relevant documents via FAISS,
3. places them in front of the question as **context**,
4. instructs the model to answer *using only that context*, and to say so if the answer is not there.

That last instruction is what keeps the model honest and grounded in your sources.

In [32]:
# Try the three retrieval strategies on the same question and compare the answers.
question = "What is the Kaitiakitanga License and what does it ensure?"

print("--- Raw query ---")
print(ask(question, strategy="raw"))
print()
print("--- With HyDE ---")
print(ask(question, strategy="hyde"))

--- Raw query ---
The Kaitiakitanga License treats data as cared for under guardianship rather than owned, and it ensures that any benefit derived from the data flows back to the people it came from.

--- With HyDE ---
Hypothetical passage: The Kaitiakitanga License is a formalized legal instrument that establishes indigenous guardianship over specific natural resources or ecosystems. It grants defined oversight authority to traditional custodians, requiring adherence to cultural mandates and sustainable resource management principles. This mechanism ensures that development activities protect ecological integrity while integrating the deep knowledge and stewardship practices of local Māori communities.
The Kaitiakitanga License treats data as cared for under guardianship rather than owned, and it ensures that any benefit derived from the data flows back to the people it came from.


## Step 9. See the difference RAG makes

Ask the same question two ways: once to the bare model (answering only from what it memorized), and once through RAG (answering from our documents). The bare model may be vague or simply make something up. This contrast is the whole point of RAG.

In [33]:
def ask_without_rag(query):
    response = ollama.chat(model=CHAT_MODEL, messages=[{"role": "user", "content": query}])
    return response["message"]["content"]

question = "What is the Kaitiakitanga License and what does it ensure?"

print("WITHOUT RAG (model answers from memory only):\n")
print(ask_without_rag(question))
print("\n" + "-" * 60 + "\n")
print("WITH RAG (model answers from our documents):\n")
print(ask(question))

WITHOUT RAG (model answers from memory only):

The term **Kaitiakitanga License** refers not necessarily to a single, universal legal document, but rather to a **commitment or formal certification structure** demonstrating that an organization, individual, or project adheres to the principles of *kaitiakitanga*.

To fully understand what it is and what it ensures, it is crucial first to understand the core concept: ***Kaitiakitanga***.

---

### 🌿 What is Kaitiakitanga? (The Principle)

*Kaitiakitanga* is a profound Māori philosophical concept originating from Aotearoa New Zealand. It translates broadly as **guardianship, stewardship, protection,** and **care**.

It is far more than simply "environmentalism." For a *kaitiaki* (guardian), this principle involves:

1.  **Ethical Responsibility:** Recognizing that humans are not the owners of the natural world, but merely temporary custodians of it.
2.  **Intergenerational Commitment:** The primary goal of guardianship is to ensure the he

## Step 10. Make it your own

This is the part that matters: point the system at *your* documents.

1. Create a folder called `my_documents` in the same place as this notebook.
2. Put some plain-text `.txt` files in it (notes, transcripts, reports, anything you want to ask questions about).
3. Run the cell below. It loads your files, splits them into overlapping chunks, saves the embeddings to an HDF5 cache file, and rebuilds the FAISS index.

Then ask away with `ask("your question here")` — try `strategy="hyde"` if your questions are phrased very differently from the documents.

In [35]:
MY_CACHE_FILE = "my_documents_cache.h5"

def load_documents_from_folder(folder, max_words=100, overlap=20):
    docs = []
    if not os.path.isdir(folder):
        print("Folder not found:", folder)
        print("Create a folder called '" + folder + "' next to this notebook, add some .txt files, then run this cell again.")
        return docs
    for name in sorted(os.listdir(folder)):
        if name.lower().endswith((".txt", ".md")):
            with open(os.path.join(folder, name), encoding="utf-8") as f:
                docs.extend(chunk_overlap(f.read(), max_words=max_words, overlap=overlap))
    print(f"Loaded {len(docs)} chunks from {folder}/")
    return docs

if os.path.exists(MY_CACHE_FILE):
    my_docs, my_embeddings = load_embeddings_hdf5(MY_CACHE_FILE)
else:
    my_docs = load_documents_from_folder("my_documents")
    if my_docs:
        print("Embedding chunks now...")
        my_embeddings = [embed(doc) for doc in my_docs]
        save_embeddings_hdf5(MY_CACHE_FILE, my_docs, my_embeddings)

if my_docs:
    # Rebuild the FAISS index and knowledge base over your documents
    documents      = my_docs
    doc_embeddings = my_embeddings
    faiss_index, normalized_matrix = build_faiss_index(doc_embeddings)
    print(f"FAISS index rebuilt: {faiss_index.ntotal} chunks ready to search.")
    print("Try: ask('what does this say about ...?')")

Loaded 45 chunks from my_documents/
Embedding chunks now...
Saved 45 embeddings to my_documents_cache.h5
FAISS index rebuilt: 45 chunks ready to search.
Try: ask('what does this say about ...?')


In [36]:
# Once you have loaded your own documents above, ask them a question here.
print(ask("What does this say about polysynthetic languages?"))

A polysynthetic language is defined as the extreme of "few, very long words," and it is simply a language where an unusually large number of meanings get packed into a single unit.

According to the text, these languages:
*   Have an asymptotic regime that would make them win on token count because one word can carry a sentence's meaning.
*   Are associated with an astronomically large *V* when covering every possible word as one token.
*   In practice, only the far-left, heavily-fragmented portion of its curve is observed.
*   The concept is that "the meanings English spreads across eight free words are crammed into one word as bound affixes," rather than being because "the units are exotic."


## Step 11. What you built, and where to go next

You just built a complete RAG system: documents in, chunking, embeddings saved to HDF5, FAISS-powered retrieval, query reformulation, and a grounded answer — all running on your own machine.

**Why the local part matters.** Nothing here used the internet after the models were downloaded. Your documents stayed on your laptop. If you were working with sensitive material, this is the difference between keeping control of it and handing it to an outside company's servers.

**A note on what you choose to put in.** A tool keeping data on your machine does not by itself answer the question of whether something *should* be digitized or modeled at all. Those are community decisions about protocols and access, and they come before the technology, not after it.

**Ways to grow this:**
- **Better chunking:** split on paragraph boundaries or headings rather than a fixed word count; experiment with smaller chunks (50 words) versus larger ones (200 words) and see how retrieval precision changes.
- **More file types:** add loaders for PDFs (`pymupdf`) or Word files (`python-docx`) instead of only `.txt`.
- **Faster FAISS at scale:** swap `IndexFlatIP` for `IndexIVFFlat` or `IndexHNSWFlat` once you have hundreds of thousands of vectors — these approximate indices trade a tiny amount of accuracy for dramatic speed gains.
- **Persistent FAISS index:** write the index to disk with `faiss.write_index(index, "index.faiss")` so you do not need to rebuild it on every restart.
- **Query expansion:** instead of one reformulated query, generate several phrasings and merge their top-k results before passing to the LLM — this is called multi-query retrieval and improves recall.
- **Different models:** run `ollama pull` to try other local models, then change `CHAT_MODEL` at the top.
- **Show your sources:** have `ask` also print which chunks it used and their scores, so answers are traceable and auditable.